In [ ]:
!pip install scanpy
!pip install igraph leidenalg
!pip install gseapy

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib as mpl
from matplotlib import pyplot as plt
import gseapy as gp

In [ ]:
#setup
path = '/Users/sienanason/Desktop/Classes/Grad_School/bc_scrnaseq'
figpath = '/Users/sienanason/Desktop/Classes/Grad_School/bc_scrnaseq/figures'

#default figure settings
sc.settings.set_figure_params(dpi_save=300, dpi=300)
sc.settings.figdir = figpath

In [ ]:
#read in cells
cells = sc.read_mtx("/Users/sienanason/Desktop/Classes/Grad_School/bc_scrnaseq/matrix.mtx").T

genes = pd.read_csv("/Users/sienanason/Desktop/Classes/Grad_School/bc_scrnaseq/genes.tsv", sep="\t", header=None)
barcodes = pd.read_csv("/Users/sienanason/Desktop/Classes/Grad_School/bc_scrnaseq/barcodes.tsv", sep="\t", header=None)

cells.var_names = genes[0].astype(str).values
cells.obs_names = barcodes[0].astype(str).values

sc.pp.filter_cells(cells, min_genes=200)
sc.pp.filter_genes(cells, min_cells=3)
cells

In [ ]:
metadata = pd.read_csv("/Users/sienanason/Desktop/Classes/Grad_School/bc_scrnaseq/metadata.csv")

# set barcodes as index
metadata = metadata.set_index("Unnamed: 0")

# attach metadata to cells
cells.obs = cells.obs.join(metadata, how="left")

cells
cells.obs.head()

In [ ]:
# save
cells.write_h5ad("/Users/sienanason/Desktop/Classes/Grad_School/bc_scrnaseq/filtered_raw.h5ad")

In [ ]:
cells.var['mt'] = cells.var_names.str.startswith('MT-')  # annotate the group of mitochondrial genes as 'mt'
sc.pp.calculate_qc_metrics(cells, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

cells.obs.head()


In [ ]:
# Visualize QC metrics as a violin plot

sc.pl.violin(cells, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True)


In [ ]:
#remove mt genes and gene counts
print(len(cells))
cells = cells[cells.obs.n_genes_by_counts > 200,]
cells = cells[cells.obs.pct_counts_mt < 10,]
print(len(cells)) 

In [ ]:
# normalize
sc.pp.normalize_total(cells)
sc.pp.log1p(cells)

In [ ]:
# save log(CPM+1) data as "raw"
cells.raw = cells

In [ ]:
# identify variable genes
sc.pp.highly_variable_genes(cells)

In [ ]:
cells.obs.head()

In [ ]:
cells

In [ ]:
# view top 10
cells.var.loc[cells.var.highly_variable].sort_values("dispersions", ascending=False).head(10)


In [ ]:
# plot variablility distribution
sc.pl.highly_variable_genes(cells)

In [ ]:
# Z-score scale. Clip values exceeding standard deviation 10.
sc.pp.scale(cells, max_value=10)

In [ ]:
# PCA
sc.tl.pca(cells)
sc.pl.pca_variance_ratio(cells, n_pcs=20)

In [ ]:
# computing the neighborhood graph and find clusters
sc.pp.neighbors(cells, n_pcs=15)
sc.tl.leiden(cells, resolution=0.05)

In [ ]:
# run UMAP
sc.tl.umap(cells)

In [ ]:
# plot UMAP
fig = sc.pl.umap(cells, color=['leiden'],
                legend_fontsize = 12,
                legend_loc = 'on data')

In [ ]:
# plot common marker genes for cells
genes_to_plot = [ 'EPCAM','KRT8',    
        'COL1A1','LUM', 
        'LYZ','C1QA',        
        'NKG7','GNLY',      
        'PECAM1','VWF'
                ]

sc.pl.umap(cells, color=genes_to_plot, use_raw=True,
                sort_order=True, ncols=3)


In [ ]:
# panels
t_cell_panel = ['CD3E','CD8A','CD4','CCR7','EOMES','KLRD1']
b_cell_panel = ['CD79A','CD19','MS4A1','JCHAIN']
monocyte_panel = ['CD14','LYZ','FCGR3A','MS4A7']
endothelial_panel = ['PECAM1','VWF','KDR']
epithelial_panel = ['EPCAM','KRT8','KRT19']
fibroblast_panel = ['COL1A1','COL1A2','LUM','DCN']
plasma_panel = ['MZB1','XBP1','PRDM1']
macrophage_panel = ['LYZ','C1QA','C1QB','CD68']
cd8_t_cell_panel = ['CD3D','CD3E','CD8A','GZMB']


In [ ]:
#0 = T-cells
#1 = Fibroblast
#2 = epithelial
#3 = CD8+ T-cells
#4 = Monocyte
#5 = endothelial cells
#6 = epithelial
#7 = plasma cells
#8 = b-cells
#9 = epithelial
sc.pl.stacked_violin(cells, cd8_t_cell_panel + epithelial_panel, groupby='leiden', swap_axes=True)

plt.show()

In [ ]:
#0 = T-cells
#1 = Fibroblast
#2 = Epithelial
#3 = CD8+ T-cells
#4 = Monocyte
#5 = endothelial cells
#6 = epithelial
#7 = plasma cells
#8 = b-cells
#9 = epithelial

new_cluster_names = ['T cells',
                    'Fibroblast',
                    'Epithelial cells1',
                    'CD8+ cytotoxic T-cells',
                    'Monocytes',
                    'Endothelial',
                    'Epithelial cells2',
                    'Plasma B cells',
                    'B cells',
                    'Epithelial cells3'
                    ]
cells.rename_categories('leiden', new_cluster_names)

In [ ]:
cells.obs['merged_clusters'] = cells.obs['leiden'].astype(str)

cells.obs.loc[
    cells.obs['merged_clusters'].isin([
        'Epithelial cells1',
        'Epithelial cells2',
        'Epithelial cells3'
    ]),
    'merged_clusters'
] = 'Epithelial cells'


# plot umap with renamed clusters
sc.pl.umap(cells, color='merged_clusters', legend_loc='on data',
          legend_fontsize=6, save='clusters_labeled.png')

In [ ]:
# Finding marker genes
# ranking for the highly differential genes in each cluster.
sc.tl.rank_genes_groups(cells, 'merged_clusters', method='wilcoxon',
                       n_genes=25)

sc.pl.rank_genes_groups(cells, n_genes=10, sharey=False,
                        fontsize=12)

In [ ]:
epi = cells[cells.obs["merged_clusters"] == "Epithelial cells"].copy()

print(epi.shape)

In [ ]:
print(epi.obs.columns.tolist())


In [ ]:
print(epi.obs["merged_clusters"].unique())

In [ ]:
# plot variablility distribution
sc.pl.highly_variable_genes(epi)

In [ ]:
# view top 10
epi.var.loc[epi.var.highly_variable].sort_values("dispersions", ascending=False).head(10)


In [ ]:
# start from epithelial cells, but recover log-normalized data from raw
epi_de = epi.raw.to_adata()

# keep the epithelial cell metadata
epi_de.obs = epi.obs.copy()

# remove ribosomal genes
epi_de = epi_de[:, ~epi_de.var_names.str.startswith(("RPL", "RPS"))].copy()

# make subtype categorical
epi_de.obs["subtype"] = epi_de.obs["subtype"].astype("category")

# DE on log-normalized, unscaled data
sc.tl.rank_genes_groups(
    epi_de,
    groupby="subtype",
    method="wilcoxon",
    use_raw=False
)

sc.pl.rank_genes_groups(
    epi_de,
    n_genes=20,
    sharey=False,
    use_raw=False
)

In [ ]:
result = epi_de.uns["rank_genes_groups"]
groups = result["names"].dtype.names

top_deg = pd.DataFrame({
    group: result["names"][group][:20]
    for group in groups
})

top_deg

In [ ]:
epi.obs["subtype"].value_counts()

In [ ]:
sc.pl.violin(
    epi,
    ["ESR1", "ERBB2", "MKI67"],
    groupby="subtype",
    save="_marker_violin.png"
)

In [ ]:
deg_tables = []

for group in groups:
    df = pd.DataFrame({
        "gene": result["names"][group],
        "logfoldchanges": result["logfoldchanges"][group],
        "pvals": result["pvals"][group],
        "pvals_adj": result["pvals_adj"][group],
        "scores": result["scores"][group],
    })
    df["subtype"] = group
    deg_tables.append(df)

deg_df = pd.concat(deg_tables, ignore_index=True)
deg_df.head()

In [ ]:
sig_deg = deg_df[(deg_df["pvals_adj"] < 0.05) & (deg_df["logfoldchanges"] > 0.25)]
sig_deg.groupby("subtype").head(20)

In [ ]:
# read the gene list
evt_genes = pd.read_csv(
    "/Users/sienanason/Desktop/Classes/Grad_School/bc_scrnaseq/EVTsubset_PC2_top400_pos.txt",
    header=None
)[0].astype(str).str.strip()

evt_genes = set(evt_genes)
len(evt_genes), list(evt_genes)[:10]

In [ ]:
result = epi_de.uns["rank_genes_groups"]
groups = result["names"].dtype.names

deg_tables = []
for group in groups:
    df = pd.DataFrame({
        "gene": result["names"][group],
        "logfoldchanges": result["logfoldchanges"][group],
        "pvals": result["pvals"][group],
        "pvals_adj": result["pvals_adj"][group],
        "scores": result["scores"][group],
    })
    df["subtype"] = group
    deg_tables.append(df)

deg_df = pd.concat(deg_tables, ignore_index=True)

In [ ]:
overlap_df = deg_df[deg_df["gene"].isin(evt_genes)].copy()
overlap_df.sort_values(["subtype", "pvals_adj", "logfoldchanges"], ascending=[True, True, False]).head(50)

In [ ]:
overlap_df["subtype"].value_counts()

In [ ]:
top_overlap = (
    overlap_df
    .sort_values(["subtype", "scores"], ascending=[True, False])
    .groupby("subtype")
    .head(20)
)

top_overlap

In [ ]:
sig_deg = deg_df[
    (deg_df["pvals_adj"] < 0.05) &
    (deg_df["logfoldchanges"] > 0.25)
].copy()

sig_overlap = sig_deg[sig_deg["gene"].isin(evt_genes)].copy()
sig_overlap.sort_values(["subtype", "logfoldchanges"], ascending=[True, False]).head(50)

In [ ]:
sig_overlap["subtype"].value_counts()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("white")
sns.set_context("paper", font_scale=1.6)

overlap_counts = sig_overlap["subtype"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(3,4))  

sns.barplot(
    x=overlap_counts.index,
    y=overlap_counts.values,
    width=0.5,
    ax=ax
)

ax.set_ylabel("Number of overlapping genes")
ax.set_xlabel("Breast cancer subtype")
ax.set_title("iEVT gene overlap with breast cancer subtypes")

sns.despine()
plt.tight_layout()

plt.savefig(
    f"{figpath}/evt_subtype_overlap_barplot.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
genes_by_subtype = (
    sig_overlap
    .groupby("subtype")["gene"]
    .apply(list)
    .to_dict()
)

In [ ]:
import gseapy as gp

enrich_results = {}

for subtype, gene_list in genes_by_subtype.items():

    enr = gp.enrichr(
        gene_list=gene_list,
        gene_sets=[
            "GO_Biological_Process_2021",
            "KEGG_2021_Human",
            "Reactome_2022"
        ],
        organism="homo sapiens",
        outdir=None
    )

    enrich_results[subtype] = enr.results

In [ ]:
enrich_results["TNBC"].head(3)

In [ ]:
top_evt = (
    sig_overlap
    .sort_values("logfoldchanges", ascending=False)
    .groupby("subtype")
    .head(15)["gene"]
)

top_evt = [g for g in top_evt if g in epi.var_names]

In [ ]:
sc.tl.score_genes(
    epi,
    gene_list=evt_genes,
    score_name="EVT_score"
)

In [ ]:
sc.pl.umap(
    epi,
    color="EVT_score",
    cmap="viridis"
)

In [ ]:
sc.pl.violin(
    epi,
    "EVT_score",
    groupby="subtype",
    save="EVT_score_subtype_breakdown.png"
)

In [ ]:
import scipy.stats as stats

groups = [epi.obs.loc[epi.obs["subtype"] == s, "EVT_score"]
          for s in epi.obs["subtype"].unique()]

stats.kruskal(*groups)

In [ ]:
sc.pl.umap(epi, color=["subtype", "EVT_score"], save="_subtype_evt_umap.png")

In [ ]:
sig_overlap["subtype"].value_counts()

In [ ]:
sc.pl.violin(
    epi,
    "EVT_score",
    groupby="leiden"
)

In [ ]:
hallmark_results = {}

for subtype, gene_list in genes_by_subtype.items():
    enr = gp.enrichr(
        gene_list=gene_list,
        gene_sets=["MSigDB_Hallmark_2020"],   # or "Hallmark_2020" if needed
        organism="homo sapiens",
        outdir=None
    )
    hallmark_results[subtype] = enr.results

In [ ]:
hallmark_results["TNBC"][["Term", "Overlap", "Adjusted P-value"]].head(10)

In [ ]:
hallmark_results["ER+"][["Term", "Overlap", "Adjusted P-value"]].head(10)

In [ ]:
hallmark_results["HER2+"][["Term", "Overlap", "Adjusted P-value"]].head(10)

In [ ]:
def plot_hallmark_bubble(df, title, top_n=15, save=None):

    plot_df = df.copy()

    plot_df = plot_df.sort_values("Adjusted P-value", ascending=True).head(top_n).copy()

    plot_df["overlap_count"] = plot_df["Overlap"].str.split("/").str[0].astype(int)
    plot_df["set_size"] = plot_df["Overlap"].str.split("/").str[1].astype(int)
    plot_df["GeneRatio"] = plot_df["overlap_count"] / plot_df["set_size"]

    plot_df = plot_df.sort_values("GeneRatio", ascending=True)

    fig, ax = plt.subplots(figsize=(9,6))

    scatter = ax.scatter(
        plot_df["GeneRatio"],
        plot_df["Term"],
        s=plot_df["overlap_count"]*120,
        c=plot_df["Adjusted P-value"],
        cmap="coolwarm_r",
        edgecolors="black",
        linewidths=0.8
    )

    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label("Adjusted P-value")

    ax.set_xlabel("GeneRatio")
    ax.set_ylabel("")
    ax.set_title(title)

    plt.tight_layout()

    if save is not None:
        plt.savefig(save, dpi=300, bbox_inches="tight")

    plt.show()

In [ ]:
plt.figure(figsize=(3,3))

plot_hallmark_bubble(
    hallmark_results["TNBC"],
    "Hallmark Enrichment TNBC Epithelial Cells Overlapping with EVT Genes",
    save=f"{figpath}/TNBC_EVT_hallmark_enrichment.png"
)

plot_hallmark_bubble(
    hallmark_results["ER+"],
    "Hallmark Enrichment ER+ Epithelial Cells Overlapping with EVT Genes",
    save=f"{figpath}/ER_EVT_hallmark_enrichment.png"
)

plot_hallmark_bubble(
    hallmark_results["HER2+"],
    "Hallmark Enrichment HER2+ Epithelial Cells Overlapping with EVT Genes",
    save=f"{figpath}/HER2_EVT_hallmark_enrichment.png"
)

In [ ]:
plot_genes = {
    "ER+": ["C15orf48", "APOD", "IGFBP2"],
    "HER2+": ["DKK1", "MT1X", "MBNL2"],
    "TNBC": ["SERPINE2", "SFRP1", "UBE2C"]
}

sc.pl.dotplot(
    epi,
    var_names=plot_genes,
    groupby="subtype",
    use_raw=True,
    standard_scale="var",
    cmap="RdBu_r"
)